# khalij-watch: point detector, run 2 (kaggle-v2)

Changes from run 1, which did not beat CFAR against GFW (F1 0.750 vs 0.800; recall on GFW detections without AIS 0.49 vs 0.78):
- detections GFW saw without AIS (`gfw_unmatched`) are positives, not ignored (index v2);
- random 96 px training crops, so vessels are not always at the chip centre;
- every epoch's weights are kept, to choose by the GFW comparison rather than our own labels.

Chips come from `khalij-watch-chips`; the index and code from `khalij-watch-index-v2`. GPU and Internet on.

Labels: Global Fishing Watch (non-commercial). Contains modified Copernicus Sentinel data.

In [ ]:
import subprocess
import sys
from pathlib import Path

found = [p.parent for p in Path('/kaggle/input').rglob('index.jsonl')]
INDEX = next(p for p in found if 'khalij-watch-index-v2' in str(p))
CHIPS = next(p for p in found if 'khalij-watch-chips' in str(p))
sys.path.insert(0, str(INDEX / 'code'))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'onnx', 'onnxruntime'], check=True)

import timm  # noqa: E402 - after the path setup above
import torch  # noqa: E402

print('index', INDEX, '| chips', CHIPS)
print('torch', torch.__version__, 'timm', timm.__version__, 'cuda', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
from training.train import Config, train

config = Config(
    index=str(INDEX / 'index.jsonl'),
    root=str(CHIPS),
    out='/kaggle/working/run',
    encoder='efficientnet_b0',
    pretrained=True,
    objectness_loss='reduced_focal',
    epochs=15,
    samples_per_epoch=40_000,
    batch_size=64,
    lr=1e-3,
    workers=4,
    crop=96,
    keep_epochs=True,
)
final = train(config)
final

In [ ]:
import json

from training.export import export_onnx, load_model, max_difference, quantise

model = load_model(Path('/kaggle/working/run/best.pt'))
fp32 = export_onnx(model, Path('/kaggle/working/point_detector.onnx'))
int8 = quantise(fp32)
report = {'fp32_max_logit_diff': max_difference(model, fp32), 'int8_max_logit_diff': max_difference(model, int8)}
Path('/kaggle/working/export.json').write_text(json.dumps(report, indent=1))
print(json.dumps(report, indent=1))
print(json.dumps(json.loads(Path('/kaggle/working/run/history.json').read_text())[-3:], indent=1))